# Stacking Ensemble with Meta-Learner for Energy Theft Detection
**Extension of:** Mohammad et al. (2023) - Ensemble-Learning-Based Decision Support System for Energy-Theft Detection in Smart-Grid Environment

## What this notebook does differently from the paper:
- The paper uses **soft voting** (averaging probabilities) as the combination strategy
- This extension uses **stacking** — the base models' predictions become features for a **meta-learner (Logistic Regression)** that learns the optimal combination
- Base models remain identical: **Random Forest (RF), XGBoost (XGB), MLP**
- Scenarios covered: **P7C** (7 classes, known consumer) and **P6C** (6 classes, known consumer)

## Architecture:
```
Input Features --> [RF | XGB | MLP] --> Out-of-Fold Predictions --> [Meta-Learner: LR] --> Final Prediction
```

## Cell 1 — Install & Import Libraries

In [ ]:
# Install any missing packages if needed
# !pip install xgboost scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, roc_curve, auc
)

# XGBoost
from xgboost import XGBClassifier

print('All libraries imported successfully!')
print(f'Pandas: {pd.__version__}, NumPy: {np.__version__}')

## Cell 2 — Load the Dataset

The TDD2022 dataset is available at Mendeley Data:
https://data.mendeley.com/datasets/c3c7329tjj/1

Download the CSV file and update `DATASET_PATH` below.

In [ ]:
# -------------------------------------------------------
# UPDATE THIS PATH to your downloaded dataset file
# -------------------------------------------------------
DATASET_PATH = '../data/Dataset_actual.csv'  # <-- change this
import os; os.makedirs('../results', exist_ok=True)

df = pd.read_csv(DATASET_PATH)

print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

## Cell 3 — Explore the Dataset

In [ ]:
print('=== Dataset Info ===')
print(df.info())

print('\n=== Target Class Distribution ===')
# The paper calls the target column 'label' or 'class' — adjust if yours differs
# Common column names: 'label', 'Label', 'class', 'Type'
TARGET_COL = 'theft'   # <-- UPDATE if your target column has a different name
CONSUMER_TYPE_COL = 'Class'  # <-- UPDATE to your consumer type column name

print(df[TARGET_COL].value_counts())

# Plot class distribution
plt.figure(figsize=(10, 4))
df[TARGET_COL].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Class Distribution in Dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cell 4 — Data Pre-Processing

Follows the paper's preprocessing steps:
1. Drop irrelevant columns
2. Handle missing values
3. Label encoding (category codes)
4. Standard scaling (mean=0, std=1)

In [ ]:
# --- Step 1: Check and handle missing values ---
print('Missing values per column:')
print(df.isnull().sum())

# Drop rows with missing values (the paper states dataset had no null values)
df.dropna(inplace=True)
print(f'\nShape after dropping NaNs: {df.shape}')

# --- Step 2: Identify feature columns ---
# The dataset has 10 consumption (numerical) features + 1 consumer_type (categorical) + 1 label
# Adjust the list below to match your actual column names

# Example: if your consumption columns are named 'h1', 'h2', ..., 'h10'
# or 'feature_1' ... 'feature_10' — update accordingly
all_cols = df.columns.tolist()
non_feature_cols = [TARGET_COL]  # exclude target

feature_cols_all = [c for c in all_cols if c not in non_feature_cols]  # 11 features (incl. consumer type)
consumption_cols = [c for c in feature_cols_all if c != CONSUMER_TYPE_COL]  # 10 consumption features only

print(f'\nAll feature columns ({len(feature_cols_all)}): {feature_cols_all}')
print(f'Consumption-only columns ({len(consumption_cols)}): {consumption_cols}')

# --- Step 3: Label encode categorical columns ---
le_consumer = LabelEncoder()
if CONSUMER_TYPE_COL in df.columns:
    df[CONSUMER_TYPE_COL] = df[CONSUMER_TYPE_COL].astype('category').cat.codes
    print(f'\nEncoded {CONSUMER_TYPE_COL} successfully.')

# Encode target
le_target = LabelEncoder()
df[TARGET_COL] = le_target.fit_transform(df[TARGET_COL])
print(f'Target classes after encoding: {le_target.classes_}')
print(f'Encoded as: {list(range(len(le_target.classes_)))}')

## Cell 5 — Prepare Scenarios: P7C and P6C

- **P7C (Protocol 7, Known Consumer):** 7 classes (Normal + Theft1-6), uses all 11 features (10 consumption + consumer type)
- **P6C (Protocol 6, Known Consumer):** 6 classes (Normal + Theft1-5, excluding Theft6 — the hardest to detect), uses all 11 features

In [ ]:
# Identify class labels after encoding
# The paper's classes: 0=Normal, 1=Theft1, 2=Theft2, 3=Theft3, 4=Theft4, 5=Theft5, 6=Theft6
# Adjust THEFT6_LABEL to the encoded integer for Theft6 in your dataset

class_names_7 = list(le_target.classes_)  # all 7 classes
print('All 7 classes:', class_names_7)

# Identify Theft6 label — it's the class named 'theft6' or 'Theft6'
# Find it programmatically:
theft6_name = [c for c in le_target.classes_ if 'theft6' in str(c).lower() or '6' in str(c)]
print('Theft6 candidate class names:', theft6_name)

# Get encoded integer for Theft6
# If theft6_name is correct, run:
# THEFT6_LABEL = le_target.transform([theft6_name[0]])[0]
# Or manually set:
THEFT6_LABEL = 6  # <-- UPDATE if different in your encoding

print(f'\nTheft6 encoded label: {THEFT6_LABEL}')

# ---- P7C: 7 classes, 11 features (known consumer) ----
X_p7c = df[feature_cols_all].values
y_p7c = df[TARGET_COL].values
class_names_p7c = [str(c) for c in le_target.classes_]

# ---- P6C: 6 classes (drop Theft6 rows), 11 features (known consumer) ----
df_p6c = df[df[TARGET_COL] != THEFT6_LABEL].copy()
X_p6c = df_p6c[feature_cols_all].values
y_p6c = df_p6c[TARGET_COL].values
# Re-encode y_p6c to be contiguous integers (0-5)
le_p6c = LabelEncoder()
y_p6c = le_p6c.fit_transform(y_p6c)
class_names_p6c = [str(c) for c in le_p6c.classes_]

print(f'\nP7C — X shape: {X_p7c.shape}, y shape: {y_p7c.shape}, classes: {np.unique(y_p7c)}')
print(f'P6C — X shape: {X_p6c.shape}, y shape: {y_p6c.shape}, classes: {np.unique(y_p6c)}')

## Cell 6 — Train/Test Split + Standard Scaling

Paper uses 80% train / 20% test split.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

# ---- P7C split ----
X_train_p7c, X_test_p7c, y_train_p7c, y_test_p7c = train_test_split(
    X_p7c, y_p7c, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_p7c
)

# ---- P6C split ----
X_train_p6c, X_test_p6c, y_train_p6c, y_test_p6c = train_test_split(
    X_p6c, y_p6c, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_p6c
)

# ---- Standard Scaler (fit on train only, transform both) ----
scaler_p7c = StandardScaler()
X_train_p7c = scaler_p7c.fit_transform(X_train_p7c)
X_test_p7c  = scaler_p7c.transform(X_test_p7c)

scaler_p6c = StandardScaler()
X_train_p6c = scaler_p6c.fit_transform(X_train_p6c)
X_test_p6c  = scaler_p6c.transform(X_test_p6c)

print('=== P7C ===')
print(f'  Train: {X_train_p7c.shape}, Test: {X_test_p7c.shape}')

print('\n=== P6C ===')
print(f'  Train: {X_train_p6c.shape}, Test: {X_test_p6c.shape}')

## Cell 7 — Define Base Models & Meta-Learner

### Base Models (same as paper):
- **Random Forest (RF):** 100 estimators, Gini criterion
- **XGBoost (XGB):** max_depth=6, eta=0.3, scale_pos_weight=1
- **MLP:** hidden_layer_sizes=(100,), ReLU, Adam, max_iter=200

### Meta-Learner (our extension):
- **Logistic Regression:** Learns to combine base model out-of-fold probability predictions
- Uses `StratifiedKFold` (5 folds) during training to generate unbiased meta-features

### Why Logistic Regression as meta-learner?
- Simple, interpretable, and well-regularized (avoids overfitting on meta-features)
- Naturally handles probability inputs from base models
- Fast to train on the meta-feature matrix

In [ ]:
def get_base_models():
    """Returns base models with paper-matching hyperparameters."""
    rf = RandomForestClassifier(
        n_estimators=100,
        criterion='gini',
        random_state=0,
        min_samples_split=2,
        min_samples_leaf=1
    )
    xgb = XGBClassifier(
        max_depth=6,
        learning_rate=0.3,         # eta=0.3
        scale_pos_weight=1,
        min_child_weight=1,
        booster='gbtree',
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        max_iter=200,
        random_state=RANDOM_STATE
    )
    return [('rf', rf), ('xgb', xgb), ('mlp', mlp)]


def get_meta_learner():
    """Returns meta-learner: Logistic Regression."""
    return LogisticRegression(
        max_iter=1000,
        solver='lbfgs',
        multi_class='multinomial',
        C=1.0,
        random_state=RANDOM_STATE
    )


def build_stacking_model():
    """Builds sklearn StackingClassifier with base models + LR meta-learner."""
    estimators = get_base_models()
    meta = get_meta_learner()
    stacking_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=meta,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        stack_method='predict_proba',  # pass probabilities to meta-learner
        passthrough=False,             # do NOT pass original features to meta-learner
        n_jobs=-1
    )
    return stacking_clf

print('Model definitions ready.')
print('\nBase models: RF, XGBoost, MLP')
print('Meta-learner: Logistic Regression (multinomial, lbfgs, C=1.0)')
print('CV strategy: StratifiedKFold (5 folds) for out-of-fold meta-features')

## Cell 8 — Helper Functions: Evaluation & Plotting

In [ ]:
def evaluate_model(model, X_test, y_test, class_names, scenario_name):
    """Computes and prints all evaluation metrics used in the paper."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    acc  = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0) * 100
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0) * 100
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0) * 100

    try:
        auc_score = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro') * 100
    except Exception:
        auc_score = float('nan')

    print(f'\n===============================')
    print(f' RESULTS: {scenario_name}')
    print(f'===============================')
    print(f'  Accuracy  : {acc:.2f}%')
    print(f'  Precision : {prec:.2f}%')
    print(f'  Recall    : {rec:.2f}%')
    print(f'  F1-Score  : {f1:.2f}%')
    print(f'  AUC Score : {auc_score:.2f}%')
    print()
    print('Per-class report:')
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    return {
        'scenario': scenario_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'auc': auc_score,
        'y_pred': y_pred,
        'y_prob': y_prob
    }


def plot_confusion_matrix(y_test, y_pred, class_names, title):
    """Plots confusion matrix matching the paper's style."""
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f'Confusion Matrix — {title}', fontsize=13)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()


def plot_roc_curve(y_test, y_prob, n_classes, class_names, title):
    """Plots one-vs-rest ROC curves for each class + macro average."""
    from sklearn.preprocessing import label_binarize
    classes = list(range(n_classes))
    y_bin = label_binarize(y_test, classes=classes)

    plt.figure(figsize=(9, 7))
    colors = plt.cm.tab10.colors

    all_fpr = np.unique(np.concatenate([roc_curve(y_bin[:, i], y_prob[:, i])[0] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i % 10], lw=1.5,
                 label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        mean_tpr += np.interp(all_fpr, fpr, tpr)

    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color='black', lw=2.5, linestyle='--',
             label=f'Macro Average (AUC = {macro_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {title}', fontsize=13)
    plt.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

    return macro_auc


print('Helper functions defined.')

## Cell 9 — Train & Evaluate Individual Base Models on P7C

Before building the stacking model, we first train and evaluate each base model independently on **P7C** — this gives us baselines to compare against the stacking model.

In [ ]:
print('Training individual base models on P7C (7 classes, known consumer)...')
print('This may take a few minutes.\n')

base_results_p7c = {}

for name, model in get_base_models():
    print(f'  Training {name.upper()}...')
    model.fit(X_train_p7c, y_train_p7c)
    y_pred = model.predict(X_test_p7c)
    acc = accuracy_score(y_test_p7c, y_pred) * 100
    f1  = f1_score(y_test_p7c, y_pred, average='macro', zero_division=0) * 100
    base_results_p7c[name] = {'accuracy': acc, 'f1': f1}
    print(f'    Accuracy: {acc:.2f}%, F1: {f1:.2f}%')

print('\nBase model training complete for P7C.')

## Cell 10 — Train Stacking Ensemble on P7C

The `StackingClassifier` internally:
1. Trains each base model on k-1 folds and generates out-of-fold probability predictions
2. Uses these OOF predictions as input features to train the Logistic Regression meta-learner
3. For final predictions: retrains base models on full training data, passes test probabilities to meta-learner

In [ ]:
import time

print('Training Stacking Ensemble (P7C)...')
print('Note: This trains 3 base models x 5 folds + 1 meta-learner. May take 30-60 min.')
print('(Wall time is comparable to the paper\'s ~36 min per experiment)')
print()

stacking_p7c = build_stacking_model()

start = time.time()
stacking_p7c.fit(X_train_p7c, y_train_p7c)
elapsed_p7c = (time.time() - start) / 60

print(f'Training complete! Wall time: {elapsed_p7c:.1f} minutes')

## Cell 11 — Evaluate Stacking Model on P7C

In [ ]:
results_p7c = evaluate_model(
    stacking_p7c,
    X_test_p7c, y_test_p7c,
    class_names=class_names_p7c,
    scenario_name='Stacking Ensemble — P7C (7 classes, Known Consumer)'
)

## Cell 12 — Confusion Matrix: P7C

In [ ]:
plot_confusion_matrix(
    y_test_p7c,
    results_p7c['y_pred'],
    class_names=class_names_p7c,
    title='Stacking Ensemble — P7C (7 classes, Known Consumer)'
)

## Cell 13 — ROC Curve: P7C

In [ ]:
macro_auc_p7c = plot_roc_curve(
    y_test_p7c,
    results_p7c['y_prob'],
    n_classes=len(class_names_p7c),
    class_names=class_names_p7c,
    title='Stacking Ensemble — P7C (7 classes, Known Consumer)'
)
print(f'Macro AUC: {macro_auc_p7c:.4f}')

## Cell 14 — Train & Evaluate Individual Base Models on P6C

In [ ]:
print('Training individual base models on P6C (6 classes, known consumer)...')
print()

base_results_p6c = {}

for name, model in get_base_models():
    print(f'  Training {name.upper()}...')
    model.fit(X_train_p6c, y_train_p6c)
    y_pred = model.predict(X_test_p6c)
    acc = accuracy_score(y_test_p6c, y_pred) * 100
    f1  = f1_score(y_test_p6c, y_pred, average='macro', zero_division=0) * 100
    base_results_p6c[name] = {'accuracy': acc, 'f1': f1}
    print(f'    Accuracy: {acc:.2f}%, F1: {f1:.2f}%')

print('\nBase model training complete for P6C.')

## Cell 15 — Train Stacking Ensemble on P6C

In [ ]:
print('Training Stacking Ensemble (P6C)...')
print()

stacking_p6c = build_stacking_model()

start = time.time()
stacking_p6c.fit(X_train_p6c, y_train_p6c)
elapsed_p6c = (time.time() - start) / 60

print(f'Training complete! Wall time: {elapsed_p6c:.1f} minutes')

## Cell 16 — Evaluate Stacking Model on P6C

In [ ]:
results_p6c = evaluate_model(
    stacking_p6c,
    X_test_p6c, y_test_p6c,
    class_names=class_names_p6c,
    scenario_name='Stacking Ensemble — P6C (6 classes, Known Consumer)'
)

## Cell 17 — Confusion Matrix: P6C

In [ ]:
plot_confusion_matrix(
    y_test_p6c,
    results_p6c['y_pred'],
    class_names=class_names_p6c,
    title='Stacking Ensemble — P6C (6 classes, Known Consumer)'
)

## Cell 18 — ROC Curve: P6C

In [ ]:
macro_auc_p6c = plot_roc_curve(
    y_test_p6c,
    results_p6c['y_prob'],
    n_classes=len(class_names_p6c),
    class_names=class_names_p6c,
    title='Stacking Ensemble — P6C (6 classes, Known Consumer)'
)
print(f'Macro AUC: {macro_auc_p6c:.4f}')

## Cell 19 — Meta-Learner Coefficient Analysis

One key advantage of using Logistic Regression as the meta-learner is interpretability — we can inspect which base models and which classes the meta-learner relies on most.

In [ ]:
def plot_meta_learner_weights(stacking_model, class_names, base_model_names, title):
    """
    Visualizes the Logistic Regression meta-learner's coefficients.
    Each base model contributes (n_classes) probability features.
    So meta-feature vector = [RF_prob_class0, ..., RF_prob_classN,
                               XGB_prob_class0, ..., XGB_prob_classN,
                               MLP_prob_class0, ..., MLP_prob_classN]
    """
    meta = stacking_model.final_estimator_
    n_classes = len(class_names)
    n_base = len(base_model_names)

    # Coefficients shape: (n_classes, n_meta_features)
    # n_meta_features = n_base_models * n_classes
    coef = meta.coef_  # shape: (n_classes, n_base * n_classes)

    # Create feature names for meta-features
    meta_feature_names = []
    for bname in base_model_names:
        for cn in class_names:
            meta_feature_names.append(f'{bname}\n{cn}')

    # Plot heatmap: rows = output classes, cols = meta-features
    fig, ax = plt.subplots(figsize=(max(14, n_base * n_classes), n_classes + 1))
    sns.heatmap(
        coef,
        annot=True, fmt='.2f', cmap='coolwarm', center=0,
        xticklabels=meta_feature_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_title(f'Meta-Learner (LR) Coefficient Heatmap — {title}', fontsize=12)
    ax.set_xlabel('Meta-Feature (base model x class)')
    ax.set_ylabel('Output Class')
    plt.tight_layout()
    plt.show()

    # Also: sum absolute coefficients per base model
    importance_per_model = {}
    for i, bname in enumerate(base_model_names):
        start_idx = i * n_classes
        end_idx   = start_idx + n_classes
        importance_per_model[bname] = np.abs(coef[:, start_idx:end_idx]).mean()

    print(f'\nAverage absolute meta-learner weight per base model ({title}):')
    for bname, imp in sorted(importance_per_model.items(), key=lambda x: -x[1]):
        print(f'  {bname.upper():5s}: {imp:.4f}')


BASE_MODEL_NAMES = ['rf', 'xgb', 'mlp']

print('=== P7C Meta-Learner Weights ===')
plot_meta_learner_weights(stacking_p7c, class_names_p7c, BASE_MODEL_NAMES, 'P7C')

print('\n=== P6C Meta-Learner Weights ===')
plot_meta_learner_weights(stacking_p6c, class_names_p6c, BASE_MODEL_NAMES, 'P6C')

## Cell 20 — Full Comparative Summary Table

Compares:
1. Individual base models (RF, XGB, MLP) — our run
2. Paper's original soft-voting ensemble results
3. **Our stacking ensemble with meta-learner**

In [ ]:
# Paper's reported results (from Table 1 in the paper)
paper_results = {
    'P7C': {
        'RF': 85.72, 'XGB': 85.67, 'MLP': 78.85,
        'Paper_Ensemble (Soft Voting)': 88.00
    },
    'P6C': {
        'RF': 94.70, 'XGB': 91.21, 'MLP': 83.27,
        'Paper_Ensemble (Soft Voting)': 94.75
    }
}

print('===================================================================')
print('         COMPARATIVE ACCURACY SUMMARY (%)')
print('===================================================================')
print(f'{"Model":<35} {"P7C":>10} {"P6C":>10}')
print('-' * 58)

# Paper results
for model_name in ['RF', 'XGB', 'MLP']:
    p7c_val = paper_results['P7C'].get(model_name, '-')
    p6c_val = paper_results['P6C'].get(model_name, '-')
    print(f'  {model_name + " (paper)":<33} {p7c_val:>10.2f} {p6c_val:>10.2f}')

print(f'  {"Paper Soft Voting Ensemble":<33} {paper_results["P7C"]["Paper_Ensemble (Soft Voting)"]:.2f}      {paper_results["P6C"]["Paper_Ensemble (Soft Voting)"]:.2f}')
print('-' * 58)

# Our base model results
for model_name in ['rf', 'xgb', 'mlp']:
    p7c_val = base_results_p7c.get(model_name, {}).get('accuracy', 0)
    p6c_val = base_results_p6c.get(model_name, {}).get('accuracy', 0)
    print(f'  {model_name.upper() + " (our run)":<33} {p7c_val:>10.2f} {p6c_val:>10.2f}')

print('-' * 58)
print(f'  {">>> Stacking Ensemble + LR (OURS)":<33} {results_p7c["accuracy"]:>10.2f} {results_p6c["accuracy"]:>10.2f}')
print('===================================================================')

# Improvement over paper
diff_p7c = results_p7c['accuracy'] - paper_results['P7C']['Paper_Ensemble (Soft Voting)']
diff_p6c = results_p6c['accuracy'] - paper_results['P6C']['Paper_Ensemble (Soft Voting)']
print(f'\nImprovement over paper\'s soft-voting ensemble:')
print(f'  P7C: {diff_p7c:+.2f}%')
print(f'  P6C: {diff_p6c:+.2f}%')

## Cell 21 — Summary Bar Chart: Accuracy Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, scenario, paper_vals, our_base_vals, stacking_acc in [
    (
        axes[0], 'P7C',
        paper_results['P7C'],
        base_results_p7c,
        results_p7c['accuracy']
    ),
    (
        axes[1], 'P6C',
        paper_results['P6C'],
        base_results_p6c,
        results_p6c['accuracy']
    )
]:
    models = ['RF\n(paper)', 'XGB\n(paper)', 'MLP\n(paper)',
              'Soft Voting\n(paper)', 'RF\n(ours)', 'XGB\n(ours)',
              'MLP\n(ours)', 'Stacking+LR\n(ours)']
    accuracies = [
        paper_vals['RF'], paper_vals['XGB'], paper_vals['MLP'],
        paper_vals['Paper_Ensemble (Soft Voting)'],
        our_base_vals['rf']['accuracy'], our_base_vals['xgb']['accuracy'],
        our_base_vals['mlp']['accuracy'], stacking_acc
    ]
    colors = ['#5b9bd5', '#5b9bd5', '#5b9bd5', '#2e75b6',
              '#70ad47', '#70ad47', '#70ad47', '#c00000']

    bars = ax.bar(models, accuracies, color=colors, edgecolor='black', width=0.6)
    ax.set_ylim(min(accuracies) - 5, 100)
    ax.set_title(f'Accuracy Comparison — {scenario}', fontsize=12)
    ax.set_ylabel('Accuracy (%)')
    ax.set_xlabel('Model')
    ax.tick_params(axis='x', labelsize=8)

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#5b9bd5', label='Paper (base models)'),
    Patch(facecolor='#2e75b6', label='Paper (soft voting ensemble)'),
    Patch(facecolor='#70ad47', label='Our run (base models)'),
    Patch(facecolor='#c00000', label='Our stacking + LR meta-learner'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.savefig('../results/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to accuracy_comparison.png')

## Cell 22 — Final Metrics Table (All Metrics, Both Scenarios)

In [ ]:
summary_data = [
    {
        'Scenario': 'P7C (7 classes, KC)',
        'Model': 'Stacking Ensemble + LR Meta-Learner',
        'Accuracy (%)': round(results_p7c['accuracy'], 2),
        'Precision (%)': round(results_p7c['precision'], 2),
        'Recall (%)': round(results_p7c['recall'], 2),
        'F1-Score (%)': round(results_p7c['f1_score'], 2),
        'AUC (%)': round(results_p7c['auc'], 2)
    },
    {
        'Scenario': 'P6C (6 classes, KC)',
        'Model': 'Stacking Ensemble + LR Meta-Learner',
        'Accuracy (%)': round(results_p6c['accuracy'], 2),
        'Precision (%)': round(results_p6c['precision'], 2),
        'Recall (%)': round(results_p6c['recall'], 2),
        'F1-Score (%)': round(results_p6c['f1_score'], 2),
        'AUC (%)': round(results_p6c['auc'], 2)
    }
]

summary_df = pd.DataFrame(summary_data)
print('=== Final Results: Stacking Ensemble with LR Meta-Learner ===')
display(summary_df)

# Save to CSV
summary_df.to_csv('../results/stacking_ensemble_results.csv', index=False)
print('\nResults saved to stacking_ensemble_results.csv')

## Cell 23 — Save Trained Models (Optional)

Save the trained stacking models to disk so you don't have to retrain.

In [ ]:
import joblib

joblib.dump(stacking_p7c, '../results/stacking_model_p7c.pkl')
joblib.dump(stacking_p6c, '../results/stacking_model_p6c.pkl')
joblib.dump(scaler_p7c, '../results/scaler_p7c.pkl')
joblib.dump(scaler_p6c, '../results/scaler_p6c.pkl')

print('Models saved:')
print('  stacking_model_p7c.pkl')
print('  stacking_model_p6c.pkl')
print('  scaler_p7c.pkl')
print('  scaler_p6c.pkl')

## Cell 24 — Reload & Predict (Optional: Load Pre-Trained Models)

Use this cell if you saved the models above and want to reload them for inference without retraining.

In [ ]:
# # Uncomment to use:

# import joblib
# stacking_p7c = joblib.load('../results/stacking_model_p7c.pkl')
# stacking_p6c = joblib.load('../results/stacking_model_p6c.pkl')
# scaler_p7c   = joblib.load('../results/scaler_p7c.pkl')
# scaler_p6c   = joblib.load('../results/scaler_p6c.pkl')
# print('Models loaded successfully!')

# # Example: predict on new data
# new_data = pd.read_csv('new_samples.csv')
# new_X = scaler_p7c.transform(new_data[feature_cols_all].values)
# predictions = stacking_p7c.predict(new_X)
# print('Predictions:', predictions)

---
## Summary

| Cell | What it does |
|------|--------------|
| 1    | Imports |
| 2    | Load dataset |
| 3    | EDA & class distribution |
| 4    | Preprocessing (encoding, scaling) |
| 5    | Build P7C and P6C scenarios |
| 6    | Train/test split + StandardScaler |
| 7    | Define base models + meta-learner + stacking builder |
| 8    | Helper functions (evaluate, confusion matrix, ROC) |
| 9    | Individual base models on P7C (baseline) |
| 10   | Train stacking ensemble on P7C |
| 11   | Evaluate stacking on P7C |
| 12   | Confusion matrix: P7C |
| 13   | ROC curve: P7C |
| 14   | Individual base models on P6C (baseline) |
| 15   | Train stacking ensemble on P6C |
| 16   | Evaluate stacking on P6C |
| 17   | Confusion matrix: P6C |
| 18   | ROC curve: P6C |
| 19   | Meta-learner coefficient analysis |
| 20   | Full comparison table (paper vs ours) |
| 21   | Bar chart comparison |
| 22   | Final metrics summary table |
| 23   | Save models |
| 24   | Reload & predict |

**Key architectural difference from the paper:**
- Paper: RF + XGB + MLP → **Soft Voting** (average probabilities, equal weight)
- Ours: RF + XGB + MLP → **Out-of-fold probability predictions** → **Logistic Regression meta-learner** (learns optimal, non-equal combination per class)
